In [3]:
import os
if os.path.basename(os.getcwd())=='notebooks':
    os.chdir('..')
import io
import pandas as pd
from google.cloud import storage
import config.settings as settings

client=storage.Client()
bucket=client.bucket(settings.BUCKET_NAME)

blobs=list(bucket.list_blobs(prefix=f"{settings.BRONZE_TRIPS_PATH}/"))
print([blob.name for blob in blobs])

['bronze/trips/', 'bronze/trips/202401-citibike-tripdata_1.csv', 'bronze/trips/202401-citibike-tripdata_2.csv']


In [6]:
file_name='bronze/trips/202401-citibike-tripdata_1.csv'
blob=bucket.blob(file_name)
df=pd.read_csv(io.BytesIO(blob.download_as_bytes()))

print(df.shape)
print(df.head())

C:\Users\Sasha\AppData\Local\Temp\ipykernel_21144\3853063292.py:3: DtypeWarning: Columns (0: start_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(io.BytesIO(blob.download_as_bytes()))


(1000000, 13)
            ride_id  rideable_type               started_at  \
0  8E865410DBDE0CA9  electric_bike  2024-01-01 13:00:04.563   
1  0403D0B3FC9CA77D  electric_bike  2024-01-08 19:36:43.520   
2  F6DE7BB42FF550BE  electric_bike  2024-01-12 15:00:41.580   
3  84A995BFD98030D4   classic_bike  2024-01-12 16:52:19.025   
4  7BBEAD4F2B535813  electric_bike  2024-01-05 19:50:19.202   

                  ended_at           start_station_name start_station_id  \
0  2024-01-01 13:04:04.652                 3 St & 3 Ave          4028.03   
1  2024-01-08 19:53:16.266  Franklin Ave & St Marks Ave          4107.05   
2  2024-01-12 15:36:29.622           W 67 St & Broadway          7116.04   
3  2024-01-12 17:17:29.773  Central Park West & W 68 St          7079.06   
4  2024-01-05 20:34:42.517           W 67 St & Broadway          7116.04   

            end_station_name end_station_id  start_lat  start_lng    end_lat  \
0      Carroll St & Smith St        4225.14  40.675070 -73.987752  40.

In [7]:
df['started_at']=pd.to_datetime(df['started_at'])
df['ended_at']=pd.to_datetime(df['ended_at'])

df['duration_sec']=(df['ended_at']-df['started_at']).dt.total_seconds()

count_travel_zero=(df['duration_sec']<=0).sum()
print(f"Длительность (<=0): {count_travel_zero}")

count_skip_stations_start=df['start_station_id'].isna().sum()
count_skip_stations_end=df['end_station_id'].isna().sum()
print(f"Пропуски start_station_id: {count_skip_stations_start}")
print(f"Пропуски end_station_id: {count_skip_stations_end}")

Длительность (<=0): 0
Пропуски start_station_id: 576
Пропуски end_station_id: 3064
